# Task 2 — OOF prediction disagreement

Read-only comparison of saved out-of-fold predictions. No fitting and no Test labels.


In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.pipeline.random_seed import set_seed
SEED = set_seed()
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import fbeta_score, mean_absolute_error
SRC = ROOT / "src"


## Matched fold inventory
All probabilities are saved OOF, with no refitting.


In [2]:
TASK = SRC / "task2"; RES = TASK / "models" / "results"
card = json.loads((TASK / "data" / "eda" / "eda_original_v2.json").read_text(encoding="utf-8"))
train = pd.read_parquet(SRC / card["splits"]["train"]["path"])
from sklearn.model_selection import train_test_split
y = train.segment.astype(int)
dev_idx, hold_idx = train_test_split(np.arange(len(y)), test_size=.2, stratify=y, random_state=SEED)
dev = train.iloc[dev_idx].reset_index(drop=True)
sources = {"baseline_cat": "class_bias_fold{i}_v1", "weighted_cat": "weighted_23_fold{i}_v4",
           "tier_cross_cat": "tier_cross_fold{i}_v5", "lightgbm": "t2_lgb_ensemble_cv5_v9_fold{i}",
           "xgboost": "t2_xgb_ensemble_cv5_v9_fold{i}"}
parts = []
for name, pattern in sources.items():
 frames = [pd.read_parquet(RES / (pattern.format(i=i) + ".parquet")) for i in range(1, 6)]
 d = pd.concat(frames, ignore_index=True).sort_values("row_pos").reset_index(drop=True)
 assert len(d) == len(dev) and np.array_equal(d.row_pos, np.arange(len(dev)))
 if "true" in d: assert np.array_equal(d.true, dev.segment)
 d["true"] = dev.segment.to_numpy()
 d["pred"] = d[[f"p_{i}" for i in range(4)]].to_numpy().argmax(axis=1)
 d["model"] = name
 parts.append(d)
oof = pd.concat(parts, ignore_index=True)
oof["wrong"] = oof.pred != oof.true
scores = (oof.groupby(["model", "fold"], sort=True)
 .apply(lambda g: fbeta_score(g.true, g.pred, beta=1, average="macro", zero_division=0), include_groups=False)
 .rename("macro_F1").reset_index())
ranking = scores.groupby("model").macro_F1.agg(["mean", "std"]).sort_values("mean", ascending=False)
ranking["accuracy"] = 1 - oof.groupby("model").wrong.mean()
display(ranking.round(5))
print("All models use 83,200 development rows and 5 matched folds; holdout and full-data model scores excluded.")


,mean,std,accuracy
model,,,
weighted_cat,0.75310,0.00352,0.76702
tier_cross_cat,0.75265,0.00313,0.76791
baseline_cat,0.75256,0.00471,0.76785
xgboost,0.74750,0.00343,0.76314
lightgbm,0.74715,0.00318,0.76246


All models use 83,200 development rows and 5 matched folds; holdout and full-data model scores excluded.


## Pairwise changes and class distribution
A changed label can still be wrong for both models.


In [3]:
def pair(a, b):
 x = oof.query("model == @a").set_index("row_pos").sort_index()
 y = oof.query("model == @b").set_index("row_pos").sort_index()
 assert x.index.equals(y.index) and x.true.equals(y.true) and x.fold.equals(y.fold)
 z = pd.DataFrame({"true": x.true, "a": x.pred, "b": y.pred,
                   "a_wrong": x.wrong, "b_wrong": y.wrong})
 z["changed"] = z.a != z.b
 z["rescue"] = z.a_wrong & ~z.b_wrong
 z["break"] = ~z.a_wrong & z.b_wrong
 z["wrong_to_wrong"] = z.a_wrong & z.b_wrong & z.changed
 return z
out = []
for a, b in [("baseline_cat", "weighted_cat"), ("baseline_cat", "tier_cross_cat"),
             ("baseline_cat", "lightgbm"), ("baseline_cat", "xgboost"),
             ("weighted_cat", "lightgbm")]:
 z = pair(a,b)
 out.append({"a":a,"b":b,"F1_a":ranking.loc[a,"mean"],"F1_b":ranking.loc[b,"mean"],
             "changed":int(z.changed.sum()), "changed_pct":100*z.changed.mean(),
             "rescued_by_b":int(z.rescue.sum()),"broken_by_b":int(z['break'].sum()),
             "wrong_to_wrong":int(z.wrong_to_wrong.sum())})
display(pd.DataFrame(out).round(4))
for a,b in [("baseline_cat","weighted_cat"),("baseline_cat","lightgbm"),("baseline_cat","xgboost")]:
 z=pair(a,b)
 print("\n",a,"vs",b,"predicted class counts")
 display(pd.DataFrame({a:z.a.value_counts().sort_index(),b:z.b.value_counts().sort_index(),
                       "true":z.true.value_counts().sort_index()}))
 print("Changed-label transition counts (rows=first model, columns=second):")
 display(pd.crosstab(z.loc[z.changed,"a"],z.loc[z.changed,"b"]))


,a,b,F1_a,F1_b,changed,changed_pct,rescued_by_b,broken_by_b,wrong_to_wrong
0,baseline_cat,weighted_cat,0.7526,0.7531,2815,3.3834,1154,1223,438
1,baseline_cat,tier_cross_cat,0.7526,0.7526,1861,2.2368,808,803,250
2,baseline_cat,lightgbm,0.7526,0.7472,5057,6.0781,1973,2421,663
3,baseline_cat,xgboost,0.7526,0.7475,4221,5.0733,1631,2023,567
4,weighted_cat,lightgbm,0.7531,0.7472,5151,6.1911,2047,2426,678



 baseline_cat vs weighted_cat predicted class counts


,baseline_cat,weighted_cat,true
0,36049,34973,32766
1,20392,19744,20825
2,15318,16553,16823
3,11441,11930,12786


Changed-label transition counts (rows=first model, columns=second):


b,0,1,2,3
a,,,,
0,0,329,1040,13
1,299,0,191,499
2,7,2,0,206
3,0,10,219,0



 baseline_cat vs lightgbm predicted class counts


... (3 outputs left out)


 baseline_cat vs xgboost predicted class counts


,baseline_cat,xgboost,true
0,36049,35893,32766
1,20392,20203,20825
2,15318,15749,16823
3,11441,11355,12786


Changed-label transition counts (rows=first model, columns=second):


b,0,1,2,3
a,,,,
0,0,650,675,6
1,811,0,106,305
2,354,63,0,427
3,10,320,494,0


## Where the weighted model changes labels
Train-derived distributions only.


In [4]:
z = pair("baseline_cat", "weighted_cat").join(dev.drop(columns="segment"), how="left")
for feature in ["loyalty_tier", "engagement_level", "vip_tier", "total_spending_thb", "days_since_last_login"]:
 vals = z[feature]
 bucket = pd.qcut(vals, 4, duplicates="drop").astype(str) if pd.api.types.is_numeric_dtype(vals) else vals.astype(str)
 part = z.assign(bucket=bucket).groupby("bucket", dropna=False).agg(
  rows=("changed","size"), changed=("changed","sum"),
  rescued=("rescue","sum"), broken=("break","sum"))
 part["changed_pct"] = part.changed / part.rows * 100
 print("\n",feature); display(part.sort_values("changed",ascending=False).head(12).round(2))
print("Weighted CatBoost class-wise F1 and recall:")
for name in ["baseline_cat","weighted_cat","lightgbm"]:
 d=oof.query("model == @name")
 print(name, "F1",np.round(fbeta_score(d.true,d.pred,beta=1,average=None,labels=[0,1,2,3],zero_division=0),4))



 loyalty_tier


,rows,changed,rescued,broken,changed_pct
bucket,,,,,
Silver,25211,1034,423,416,4.10
Gold,13127,847,322,366,6.45
None,6401,435,172,193,6.80
Bronze,25061,310,141,159,1.24
Platinum,8057,108,62,45,1.34
Diamond,5343,81,34,44,1.52



 engagement_level


,rows,changed,rescued,broken,changed_pct
bucket,,,,,
Mid,30712,1180,473,509,3.84
Low,26029,738,317,320,2.84
High,16893,568,218,261,3.36
None,6506,279,117,115,4.29
Veteran,3060,50,29,18,1.63



 vip_tier


,rows,changed,rescued,broken,changed_pct
bucket,,,,,
"(-0.001, 1.0]",49191,1454,596,624,2.96
"(2.0, 5.0]",15759,637,265,276,4.04
"(1.0, 2.0]",11868,471,192,203,3.97
nan,6382,253,101,120,3.96



 total_spending_thb


,rows,changed,rescued,broken,changed_pct
bucket,,,,,
"(49174.777, 140823.228]",19198,733,299,316,3.82
"(31730.269, 49174.777]",19198,719,301,301,3.75
"(16395.092, 31730.269]",19198,597,242,268,3.11
"(501.185, 16395.092]",19199,549,218,252,2.86
nan,6407,217,94,86,3.39



 days_since_last_login


,rows,changed,rescued,broken,changed_pct
bucket,,,,,
"(31.282, 45.763]",19195,684,286,276,3.56
"(-0.0008320000000000001, 31.282]",19195,676,271,304,3.52
"(45.763, 60.154]",19194,668,265,291,3.48
"(60.154, 132.869]",19195,567,249,249,2.95
nan,6421,220,83,103,3.43


Weighted CatBoost class-wise F1 and recall:
baseline_cat F1 [0.8215 0.7412 0.71   0.7376]
weighted_cat F1 [0.8201 0.7397 0.7121 0.7405]
lightgbm F1 [0.818  0.736  0.7051 0.7295]


## Saved development-fit models on reused holdout
For tuned candidates without saved OOF rows, compare predictions on the repeatedly consulted holdout. Full-data refits are excluded.


In [5]:
import joblib
hold = train.iloc[hold_idx].copy()
hold_names = ["t2_cat_original_no_noise_v2", "t2_cat_weighted_23_cv5_v4",
              "t2_cat_tuned_cv5_v6", "t2_cat_tuned_weighted_cv5_v7"]
hold_pred = {}
for name in hold_names:
 model = joblib.load(TASK / "save" / f"{name}.joblib")
 columns = list(model.feature_names_in_)
 hold_pred[name] = np.asarray(model.predict(hold[columns])).reshape(-1).astype(int)
 result = json.loads((TASK / "save" / f"{name}-evaluation.json").read_text(encoding="utf-8"))
 score = fbeta_score(hold.segment, hold_pred[name], beta=1, average="macro")
 assert result["metric"] == "macro_F1" and abs(result["score"]-score)<1e-9
 print(name, "reused holdout F1", round(score,6))
for a,b in [(hold_names[0],hold_names[1]),(hold_names[2],hold_names[3]),
            (hold_names[0],hold_names[3])]:
 pa,pb=hold_pred[a],hold_pred[b]
 changed=pa!=pb
 print(a,"vs",b,"changed",int(changed.sum()),"of",len(hold),
       "rescue",int(((pa!=hold.segment)&(pb==hold.segment)).sum()),
       "break",int(((pa==hold.segment)&(pb!=hold.segment)).sum()))
 display(pd.DataFrame({a:pd.Series(pa).value_counts().sort_index(),
                       b:pd.Series(pb).value_counts().sort_index(),
                       "true":hold.segment.value_counts().sort_index().to_numpy()}))


t2_cat_original_no_noise_v2 reused holdout F1 0.750414


t2_cat_weighted_23_cv5_v4 reused holdout F1 0.752849


t2_cat_tuned_cv5_v6 reused holdout F1 0.751095


t2_cat_tuned_weighted_cv5_v7 reused holdout F1 0.752895
t2_cat_original_no_noise_v2 vs t2_cat_weighted_23_cv5_v4 changed 659 of 20800 rescue 293 break 271


,t2_cat_original_no_noise_v2,t2_cat_weighted_23_cv5_v4,true
0,9023,8767,8191
1,5169,5019,5207
2,3784,4075,4206
3,2824,2939,3196


t2_cat_tuned_cv5_v6 vs t2_cat_tuned_weighted_cv5_v7 changed 601 of 20800 rescue 261 break 255


,t2_cat_tuned_cv5_v6,t2_cat_tuned_weighted_cv5_v7,true
0,8944,8731,8191
1,5197,5056,5207
2,3812,4045,4206
3,2847,2968,3196


t2_cat_original_no_noise_v2 vs t2_cat_tuned_weighted_cv5_v7 changed 725 of 20800 rescue 322 break 303


,t2_cat_original_no_noise_v2,t2_cat_tuned_weighted_cv5_v7,true
0,9023,8731,8191
1,5169,5056,5207
2,3784,4045,4206
3,2824,2968,3196


## Assessment

- **Medium priority backup:** tuned unweighted CatBoost. Its paired CV gap to tuned weighted CatBoost is only 0.000061 F₁; the reused holdout gap is larger and exploratory.
- **Medium priority:** older weighted CatBoost. It changes 2,815 of 83,200 OOF labels versus baseline, moving predictions toward Social and Whale, while rescuing 1,154 and breaking 1,223 rows.
- **Low priority:** LightGBM/XGBoost diversity is real (5,057/4,221 switches against CatBoost), but their F₁ is lower and previously tested fixed ensembles lost.
